# 📗 Notebook 2 — RAG with LangChain v1 (No Memory, No UI)
### Same Pipeline · LangChain LCEL · Groq · HuggingFace Embeddings (Free)

**What you will see:** The same RAG pipeline from Notebook 1 — rebuilt using
LangChain v1's modern LCEL approach. Compare the code and see what LangChain saves.

**Key change from the uploaded notebook:** Uses Groq (free) instead of OpenAI,
and HuggingFace embeddings (free, local) instead of paid OpenAI embeddings.

**What LangChain v1 changed from v0:**

| Old (broken in v1) | New (correct in v1) |
|---|---|
| `from langchain.chains import RetrievalQA` | Use LCEL pipeline instead |
| `from langchain.schema import Document` | `from langchain_core.documents import Document` |
| `from langchain.text_splitter import ...` | `from langchain_text_splitters import ...` |
| `chain.run(question)` | `chain.invoke(question)` |

**LCEL Pipeline:**
```
question → retriever → format_docs → prompt_template → llm → StrOutputParser → answer
```


In [ ]:
# ── CELL 1: Install ──────────────────────────────────────────────────────────
!pip install langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers -q
print("✅ All packages installed")


In [ ]:
# ── CELL 2: Set API Key ──────────────────────────────────────────────────────
import os
os.environ["GROQ_API_KEY"] = "enter-your-api-key-here"
print("✅ API key set")


In [ ]:
# ── CELL 3: Imports — LangChain v1 Correct Imports ──────────────────────────
# IMPORTANT: These imports are for LangChain v1.
# The old langchain.chains.RetrievalQA is GONE in v1.
# We use LCEL (pipe operator) instead.

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os

print("✅ All LangChain v1 imports successful")


In [ ]:
# ── CELL 4: Initialize LLM and Embeddings ────────────────────────────────────
# LLM: Groq's LLaMA 3.3 70B via LangChain's ChatGroq wrapper
# Embeddings: HuggingFace all-MiniLM-L6-v2 — FREE, runs locally

# Groq LLM — same model as Notebook 1, but via LangChain interface
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    groq_api_key=os.getenv("GROQ_API_KEY"),
)
print("✅ ChatGroq initialized: openai/gpt-oss-120b")

# HuggingFace Embeddings — same model as Notebook 1
# LangChain wraps it so the API is consistent regardless of which embedding model you use
print("⏳ Loading HuggingFace embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("✅ HuggingFaceEmbeddings ready: all-MiniLM-L6-v2")
print()
print("COMPARISON: In Notebook 1 we called embed_model.encode() directly.")
print("Here, LangChain wraps it — the FAISS vector store calls it automatically.")


In [ ]:
# ── CELL 5: Documents ────────────────────────────────────────────────────────
# Same content as Notebook 1, but wrapped in LangChain Document objects.
# Document has .page_content and .metadata — this is the standard format
# used by all LangChain loaders and vector stores.

raw_texts = [
    """RAG stands for Retrieval-Augmented Generation.
It is a technique where an AI model retrieves relevant documents before generating an answer.
RAG helps the model answer questions about data it was not trained on.
This reduces hallucinations and keeps answers grounded in real content from your own documents.
The two phases are: indexing (done once) and retrieval+generation (done on every query).""",

    """LangChain is a framework for developing applications powered by language models.
It provides tools to connect LLMs with external data sources and APIs.
In LangChain v1, legacy chains were removed from the core package.
The modern approach uses LCEL (LangChain Expression Language) to build pipelines.
LCEL uses the pipe operator | to chain components together.
LangChain provides pre-built loaders, splitters, embeddings, and vector store wrappers.""",

    """FAISS is a library developed by Facebook AI for efficient similarity search.
It is commonly used as a vector store in RAG pipelines.
FAISS stores embedding vectors and retrieves the most similar ones for a given query.
IndexFlatIP performs exact inner product (cosine) similarity search.
For large datasets, IndexHNSWFlat provides approximate but much faster search.
FAISS runs entirely in memory — there is no server or database to set up.""",

    """An embedding is a list of numbers that represents the meaning of text.
Sentences with similar meanings produce similar vectors.
The all-MiniLM-L6-v2 model produces 384-dimensional vectors for free.
OpenAI's text-embedding-3-small produces 1536-dimensional vectors for a small cost.
Cosine similarity measures how close two embeddings are: 1.0 = identical, 0.0 = unrelated.
The embedding model used for indexing and querying must always be the same model.""",

    """Groq provides fast inference for open-source models like LLaMA and Gemma.
The Groq API is compatible with the OpenAI client — just change the base_url.
LLaMA 3.3 70B is a powerful open-source model available free on Groq's tier.
Response times on Groq are extremely fast due to their custom LPU hardware.
Get a free API key at console.groq.com/keys — no credit card required.""",
]

# Wrap in LangChain Document objects with metadata
documents = [
    Document(page_content=text, metadata={"source": f"doc_{i+1}"})
    for i, text in enumerate(raw_texts)
]

print(f"✅ {len(documents)} LangChain Document objects created")
print(f"   Each has .page_content and .metadata")
print(f"   Example metadata: {documents[0].metadata}")


In [ ]:
# ── CELL 6: Split Documents ──────────────────────────────────────────────────
# RecursiveCharacterTextSplitter is smarter than our raw chunk_text():
# It tries to split on: paragraphs → sentences → words → characters
# This means it never cuts mid-sentence (our raw version sometimes did)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(f"✅ Split into {len(chunks)} chunks")
print(f"\nExample chunk:")
print(f"  Content: {chunks[0].page_content[:120]}...")
print(f"  Metadata: {chunks[0].metadata}")
print()
print("COMPARISON vs Notebook 1:")
print("  Notebook 1: chunk_text() — manual loop, splits on character count")
print("  Notebook 2: RecursiveCharacterTextSplitter — splits on natural boundaries")


In [ ]:
# ── CELL 7: Build FAISS Vector Store ─────────────────────────────────────────
# One line replaces all of build_index() from Notebook 1.
# LangChain's FAISS wrapper handles: embedding all chunks + building the index.

vector_store = FAISS.from_documents(chunks, embeddings)

print(f"✅ FAISS vector store created")
print(f"   Vectors stored: {vector_store.index.ntotal}")
print()
print("COMPARISON vs Notebook 1:")
print("  Notebook 1: ~20 lines — manually embed chunks, build IndexFlatIP, add vectors")
print("  Notebook 2: 1 line — FAISS.from_documents() does everything")


In [ ]:
# ── CELL 8: Create Retriever ─────────────────────────────────────────────────
# A retriever is a standard interface for fetching relevant documents.
# By using as_retriever(), we get an object that can be plugged directly
# into LCEL chains with the | operator.

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Quick test
test_docs = retriever.invoke("What is RAG?")
print(f"✅ Retriever created (k=3 — fetches top 3 most relevant chunks)")
print(f"\nTest retrieval for 'What is RAG?':")
print(f"  Retrieved {len(test_docs)} chunks")
print(f"  Chunk 1: {test_docs[0].page_content[:100]}...")
print()
print("COMPARISON vs Notebook 1:")
print("  Notebook 1: retrieve() function — manually embed query, search FAISS, format results")
print("  Notebook 2: retriever.invoke() — one call, standard interface, pluggable into chains")


In [ ]:
# ── CELL 9: Define Prompt Template ───────────────────────────────────────────
# ChatPromptTemplate creates a reusable prompt with named variables.
# {context} and {question} will be filled automatically by the chain.
# This is cleaner than f-strings and supports multiple formats.

prompt_template = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer ONLY using the context provided below.
If the answer is not in the context, say exactly: "I don't have that information."
Do NOT make up any facts. Be concise and clear.

Context:
{context}

Question: {question}

Answer:
""")

print("✅ ChatPromptTemplate defined")
print("   Variables: {context} and {question}")
print()
print("COMPARISON vs Notebook 1:")
print("  Notebook 1: f-string prompt built manually in generate_answer()")
print("  Notebook 2: ChatPromptTemplate — reusable, testable, version-controllable")


In [ ]:
# ── CELL 10: Build the LCEL RAG Chain ────────────────────────────────────────
# LCEL (LangChain Expression Language) uses | to chain components.
# This replaces the old RetrievalQA chain that no longer exists in v1.
#
# Pipeline:
# question → {context: retrieve+format, question: passthrough}
#          → prompt_template
#          → llm
#          → StrOutputParser (extracts string from LLM response)

def format_docs(docs):
    """Join retrieved Document objects into a single context string."""
    return "\n\n".join(
        f"[Chunk {i+1}]\n{doc.page_content}"
        for i, doc in enumerate(docs)
    )


rag_chain = (
    {
        "context":  retriever | format_docs,   # retrieve chunks, then format them
        "question": RunnablePassthrough()       # pass the question through unchanged
    }
    | prompt_template     # fill {context} and {question} into the template
    | llm                 # call ChatGroq (LLaMA 3.3 70B)
    | StrOutputParser()   # extract the string content from the LLM response
)

print("✅ RAG chain built using LCEL")
print()
print("Chain structure:")
print("  question")
print("    ↓")
print("  {context: retriever | format_docs, question: RunnablePassthrough()}")
print("    ↓")
print("  prompt_template  (fills {context} and {question})")
print("    ↓")
print("  llm  (ChatGroq — LLaMA 3.3 70B)")
print("    ↓")
print("  StrOutputParser()  (extracts answer string)")
print("    ↓")
print("  answer")


In [ ]:
# ── CELL 11: ask() helper + Test Questions ───────────────────────────────────

def ask(question: str, show_sources: bool = True):
    """Ask a question through the LCEL RAG chain and display results."""
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print("-"*60)

    answer = rag_chain.invoke(question)
    print(f"ANSWER:\n{answer}")

    if show_sources:
        retrieved = retriever.invoke(question)
        print(f"\nSOURCE CHUNKS ({len(retrieved)} retrieved):")
        for i, doc in enumerate(retrieved, 1):
            print(f"  [{i}] {doc.page_content[:90]}...")

    print(f"{'='*60}\n")


# Run all questions
ask("What is RAG and why is it useful?")


In [ ]:
ask("What changed in LangChain v1 regarding chains?")


In [ ]:
ask("What is FAISS and where does it come from?")


In [ ]:
# This question is outside our documents — should say it doesn't know
ask("Who is the CEO of Groq?")


In [ ]:
# ── CELL 12: The Problem — Try a Follow-Up ───────────────────────────────────
# This demonstrates WHY we need memory (Notebook 3)

print("DEMONSTRATING THE MEMORY PROBLEM")
print("="*60)
print()

ask("What is RAG?", show_sources=False)

print("NOW: Ask a follow-up that references the previous answer:")
print()

ask("Tell me more about the indexing phase you just mentioned.", show_sources=False)

print()
print("OBSERVE: The second answer may not connect to the first answer.")
print("The chain does not know what 'you just mentioned' refers to.")
print("Each rag_chain.invoke() call is completely independent.")
print()
print("SOLUTION: Notebook 3 adds conversation memory to fix this.")
